## 0. Environment

In [18]:
# ==============================
# Initialization: Environment & Backends
# ==============================

# If needed:
# %pip install -q accelerate>=0.34.2 safetensors>=0.4.3 sentencepiece packaging

import os, random, numpy as np, torch

# ---- Repro ----
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# ---- Device & backend ----
device  = "cuda" if torch.cuda.is_available() else "cpu"
BACKEND = os.environ.get("LLM_BACKEND", "hf")  # "hf" or "openai"

# ---- Models ----
HF_MODEL   = os.environ.get("HF_MODEL", "TinyLlama/TinyLlama-1.1B-Chat-v1.0")
GPT5_MODEL = os.environ.get("GPT5_MODEL", "gpt-5")
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")

print(f"Device: {device}")
print(f"Backend: {BACKEND}")
print(f"HF Model: {HF_MODEL if BACKEND=='hf' else 'N/A'}")
print(f"GPT-5 Model: {GPT5_MODEL if BACKEND=='openai' else 'N/A'}")

# GTX 1080 Ti notes: no bfloat16, SDPA/flash attn can be flaky -> force eager attention
ATTN_IMPL = "eager"  # safe default for older GPUs

if BACKEND == "hf":
    from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer

    # Load config first to pin attention implementation
    config = AutoConfig.from_pretrained(HF_MODEL)
    try:
        # Not every config exposes this, so guard it
        setattr(config, "attn_implementation", ATTN_IMPL)
    except Exception:
        pass

    tokenizer = AutoTokenizer.from_pretrained(HF_MODEL, use_fast=True)
    model = AutoModelForCausalLM.from_pretrained(
        HF_MODEL,
        config=config,
        torch_dtype=torch.float16 if device == "cuda" else torch.float32,
        low_cpu_mem_usage=True,
        trust_remote_code=False,   # flip to True only if your model requires it
    )
    # Pad token safety
    if tokenizer.pad_token is None and tokenizer.eos_token is not None:
        tokenizer.pad_token = tokenizer.eos_token

    # Move to device
    model.to(device)
    model.eval()
    print("✅ Loaded HF model (fp16 on CUDA if available), attention=eager.")
else:
    import openai
    if not OPENAI_API_KEY:
        raise EnvironmentError("OPENAI_API_KEY not set; export it to use the OpenAI backend.")
    openai.api_key = OPENAI_API_KEY
    print(f"✅ OpenAI backend ready (model={GPT5_MODEL}). temperature will be set to 0.0 in calls.")


Device: cuda
Backend: openai
HF Model: N/A
GPT-5 Model: gpt-5
✅ OpenAI backend ready (model=gpt-5). temperature will be set to 0.0 in calls.
